<a href="https://colab.research.google.com/github/eeeewyz/agent/blob/main/8_write%20codeplan_execution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M5 Agentic AI - Customer Service Agent

## 1. Introduction

As Andrew explained in the lecture, *planning with code execution* means letting the LLM **write code that becomes the plan itself**.  
Compared to plain-text or JSON-based plans, this approach is more expressive and flexible: the code not only documents the steps but can also execute them directly.

In this lab, you will implement this design pattern in practice.  
Instead of asking the LLM to output a plan in JSON format and then manually executing each step, we will allow it to **write Python code** that directly captures multiple steps of a plan. By executing this code, we can carry out complex queries automatically.  

To make things concrete, we simulate a **sunglasses store** with an **inventory** of products and a set of **transactions** (sales, returns, balance updates). This example shows how the LLM can generate code to query or update records, demonstrating the flexibility of this pattern.

### 1.1 Lab Overview
We will:
1. Create simple **inventory** and **transaction** datasets.  
2. Build a **schema block** describing the data.  
3. Prompt the LLM to **write a plan as Python code** (with comments explaining each step).  
4. Execute the code in a sandbox to obtain the answer.  

### 1.2 Learning Outcomes

By the end of this lab, you will be able to:

- **Explain** why letting the model write code (instead of JSON or plain text plans) enables richer, more flexible planning.  
- **Prompt** an LLM to produce Python code with step-by-step comments that both documents and executes the plan.  
- **Run** the generated code safely in a sandbox and interpret the results.  

This illustrates how *Code as Action* can outperform brittle tool chains and JSON-based planning approaches.

## 2. Setup

In [ ]:
# ==== Imports ====
from __future__ import annotations
import json
from dotenv import load_dotenv
from openai import OpenAI
import re, io, sys, traceback, json
from typing import Any, Dict, Optional
from tinydb import Query, where

# Utility modules
import utils      # helper functions for prompting/printing
import inv_utils  # functions for inventory, transactions, schema building, and TinyDB seeding

load_dotenv()
client = OpenAI()

In the `inv_utils` module, we have functions like:

- `create_inventory()` – builds the sunglasses inventory.  
- `create_transactions()` – builds the initial transaction log.  
- `seed_db()` – loads both inventory and transactions into a JSON-backed store.  
- `build_schema_block()` – generates a schema description used in the prompt.  
- Helpers like `get_current_balance()` and `next_transaction_id()` – let the LLM handle consistent updates across inventory and transactions.  

### 2.1 Create Example Tables

We will now create two small tables for the sunglasses store simulation, using **[TinyDB](https://tinydb.readthedocs.io/)** — a lightweight document-oriented database written in pure Python.  
TinyDB stores data as JSON documents and is well-suited for small applications or prototypes, since it requires no server setup and allows you to query and update data easily.

The two tables are:

- **`inventory_tbl`**: contains product details such as name, item ID, description, quantity in stock, and price.  
- **`transactions_tbl`**: starts with an opening balance and will later track purchases, returns, and adjustments.  

You will generate these tables using helper functions in `inv_utils`, and then preview the first few rows below.

使用 TinyDB 创建一个模拟太阳镜商店的小型数据库环境，供后面的 Agent 查询和操作。

拆开：

create two small tables
创建两个小表：
inventory_tbl：库存表（商品信息、价格、数量等）
transactions_tbl：交易表（购买、退货、调整记录等）


using TinyDB
使用 TinyDB 作为数据库。
TinyDB 是一个轻量级的 Python 文档数据库。


stores data as JSON documents
数据不是像传统 SQL 表格那样存储，而是以 JSON 文档形式保存。

目的：初始化数据环境。

创建一个 TinyDB 数据库
创建两个表：
inventory_tbl：存放商品库存信息
transactions_tbl：存放交易记录

也就是为后面的 Agent 查询、更新数据准备一个数据库。

In [ ]:
db, inventory_tbl, transactions_tbl = inv_utils.seed_db()

目的：查看刚创建的表里面有什么数据。

Now, you can inspect the records in each table by printing them as formatted JSON:

In [ ]:
utils.print_html(json.dumps(inventory_tbl.all(), indent=2), title="Inventory Table")
utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table")

As you can see above, the schemas of each table are as follows:

<div style="border:1px solid #BFDBFE; border-left:6px solid #3B82F6; background:#EFF6FF; border-radius:6px; padding:16px; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif; line-height:1.6; color:#1E3A8A;">

  <h4 style="margin-top:0; color:#1E40AF;">Inventory Table (<code>inventory_tbl</code>)</h4>
  <ul>
    <li><strong>item_id</strong> (string): Unique product identifier (e.g., SG001).</li>
    <li><strong>name</strong> (string): Style of sunglasses (e.g., Aviator, Round).</li>
    <li><strong>description</strong> (string): Text description of the product.</li>
    <li><strong>quantity_in_stock</strong> (int): Current stock available.</li>
    <li><strong>price</strong> (float): Price in USD.</li>
  </ul>
  <h4 style="margin-top:1em; color:#1E40AF;">Transactions Table (<code>transactions_tbl</code>)</h4>
  <ul>
    <li><strong>transaction_id</strong> (string): Unique identifier (e.g., TXN001).</li>
    <li><strong>customer_name</strong> (string): Name of the customer, or <code>OPENING_BALANCE</code> for initial entry.</li>
    <li><strong>transaction_summary</strong> (string): Short description of the transaction.</li>
    <li><strong>transaction_amount</strong> (float): Amount of money for this transaction.</li>
    <li><strong>balance_after_transaction</strong> (float): Running balance after applying the transaction.</li>
    <li><strong>timestamp</strong> (string): ISO-8601 formatted date/time of the transaction.</li>
  </ul>
</div>


## Planning with Code Execution

### 2.1. The plan

Once the schema is clear, you’ll build the **prompt** that instructs the model to *plan by writing code* and then execute that code. As Andrew emphasized, the code is the plan: the model explains each step in comments, then carries it out. Your prompt below also makes the model self-decide whether the request is read-only or a state change, and it enforces safe execution (no I/O, no network, TinyDB Query only, consistent mutations).


用户问题
   ↓
LLM分析
   ↓
生成Python代码
   ↓
Python执行
   ↓
操作TinyDB
   ↓
answer_text返回用户

所以下方这个是：System-level instruction / Agent prompt

作用：

告诉 LLM：

你的任务是什么
可以使用什么工具
如何规划
如何生成代码
有哪些安全限制

PROMPT = """
你是一名高级数据助手（Senior Data Assistant）。
你的任务是：通过编写 Python 代码来规划并执行操作，使用 TinyDB 数据库完成用户请求。

数据库结构和示例数据（只读）：
{schema_block}


==============================
执行环境（已经导入并提供）
==============================

你可以直接使用：

- db:
    TinyDB 数据库对象

- inventory_tbl:
    库存表（商品信息）

- transactions_tbl:
    交易记录表

辅助函数：

- get_current_balance(tbl) -> float
    获取当前余额

- next_transaction_id(tbl, prefix="TXN") -> str
    生成新的交易 ID

用户请求：

- user_request: str
    用户原始输入


==============================
规划规则（重要）
==============================

你的任务不是直接回答用户。

你必须：

1. 分析用户请求
2. 根据请求生成 Python 代码
3. 执行代码完成查询或数据库修改


规则：

- 所有过滤条件必须从用户请求中提取。
- 不允许硬编码固定条件。

例如：

错误：

price < 100


正确：

根据用户输入：

"100美元以下"

动态生成：

Query().price < 100


需要解析的信息包括：

- 商品类型
- 关键词
- 价格范围：
    under / over / between
- 库存要求
- 数量
- 购买 / 退货意图


如果用户意图不明确：

默认执行只读操作（DRY RUN），不要修改数据库。


==============================
交易处理规则（严格）
==============================

禁止创建一个包含多个商品的聚合交易。


如果用户购买多个商品：

必须：

每个商品创建一条独立 transaction。


对于每个商品：

1. 计算自己的总价：

unit_price × quantity


2. 插入一条交易记录。


3. 更新余额：

balance += line_total


4. 更新库存数量。


如果任意商品库存不足：

禁止修改任何数据。


返回：

STATUS="insufficient_stock"


==============================
用户回复要求
==============================

必须设置：

answer_text


类型：

str


它是唯一返回给用户的信息。


要求：

- 简短
- 友好
- 1～2句话


禁止返回：

- JSON
- dataframe
- 技术日志


例如：

成功：

"Yes, we have Classic sunglasses available for $60."


没有匹配：

"We don't have this style now, but we have a similar model."


==============================
操作判断规则
==============================


如果用户明确要求修改状态：

例如：

- buy
- purchase
- return
- restock
- adjust


执行：

ACTION="mutate"

SHOULD_MUTATE=True


并修改数据库。


否则：

ACTION="read"

SHOULD_MUTATE=False


只查询数据，并模拟执行（dry run）。


==============================
异常处理
==============================

必须设置：

STATUS


只能是以下之一：


"success"

操作成功


"no_match"

没有找到符合条件的商品


"insufficient_stock"

商品存在，但库存不足


"invalid_request"

用户请求缺少必要信息

例如：

"我要买"

但没有数量。


"unsupported_intent"

超出商店能力范围


所有情况下：

保持回复简洁友好。


技术信息：

例如：

ACTION

DRY_RUN

STATUS


只能输出到日志。


==============================
代码要求
==============================


生成 Python 代码时：

必须：

1. 解析用户请求

可以使用 regex。


2. 使用 TinyDB Query 动态构建查询。


3. 如果是修改操作：

执行：

- 检查库存
- 更新库存
- 创建 transaction
- 更新余额


4. 始终设置：

answer_text

STATUS


5. 可以选择设置：

answer_rows

answer_json


但：

answer_text 必须存在。


==============================
输出格式
==============================


只允许输出 Python 代码。


必须放在：

<execute_python>

你的 Python 代码

</execute_python>


禁止输出任何解释文字。


==============================
代码规范
==============================


- 使用 TinyDB Query 查询。
- 必要时只能使用标准库。
- 保持代码清晰。
- 使用编号注释说明步骤。


==============================
用户请求
==============================

{question}

"""

In [ ]:
PROMPT = """You are a senior data assistant. PLAN BY WRITING PYTHON CODE USING TINYDB.

Database Schema & Samples (read-only):
{schema_block}

Execution Environment (already imported/provided):
- Variables: db, inventory_tbl, transactions_tbl  # TinyDB Table objects
- Helpers: get_current_balance(tbl) -> float, next_transaction_id(tbl, prefix="TXN") -> str
- Natural language: user_request: str  # the original user message

PLANNING RULES (critical):
- Derive ALL filters/parameters from user_request (shape/keywords, price ranges "under/over/between", stock mentions,
  quantities, buy/return intent). Do NOT hard-code values.
- Build TinyDB queries dynamically with Query(). If a constraint isn't in user_request, don't apply it.
- Be conservative: if intent is ambiguous, do read-only (DRY RUN).

TRANSACTION POLICY (hard):
- Do NOT create aggregated multi-item transactions.
- If the request contains multiple items, create a separate transaction row PER ITEM.
- For each item:
  - compute its own line total (unit_price * qty),
  - insert ONE transaction with that amount,
  - update balance sequentially (balance += line_total),
  - update the item’s stock.
- If any requested item lacks sufficient stock, do NOT mutate anything; reply with STATUS="insufficient_stock".

HUMAN RESPONSE REQUIREMENT (hard):
- You MUST set a variable named `answer_text` (type str) with a short, customer-friendly sentence (1–2 lines).
- This sentence is the only user-facing message. No dataframes/JSON, no boilerplate disclaimers.
- If nothing matches, politely say so and offer a nearby alternative (closest style/price) or a next step.

ACTION POLICY:
- If the request clearly asks to change state (buy/purchase/return/restock/adjust):
    ACTION="mutate"; SHOULD_MUTATE=True; perform the change and write a matching transaction row.
  Otherwise:
    ACTION="read"; SHOULD_MUTATE=False; simulate and explain briefly as a dry run (in logs only).

FAILURE & EDGE-CASE HANDLING (must implement):
- Do not capture outer variables in Query.test. Pass them as explicit args.
- Always set a short `answer_text`. Also set a string `STATUS` to one of:
  "success", "no_match", "insufficient_stock", "invalid_request", "unsupported_intent".
- no_match: No items satisfy the filters → suggest the closest in style/price, or invite a different range.
- insufficient_stock: Item found but stock < requested qty → state available qty and offer the max you can fulfill.
- invalid_request: Unable to parse essential info (e.g., quantity for a purchase/return) → ask for the missing piece succinctly.
- unsupported_intent: The action is outside the store’s capabilities → provide the nearest supported alternative.
- In all cases, keep the tone helpful and concise (1–2 sentences). Put technical details (e.g., ACTION/DRY RUN) only in stdout logs.

OUTPUT CONTRACT:
- Return ONLY executable Python between these tags (no extra text):
  <execute_python>
  # your python
  </execute_python>

CODE CHECKLIST (follow in code):
1) Parse intent & constraints from user_request (regex ok).
2) Build TinyDB condition incrementally; query inventory_tbl.
3) If mutate: validate stock, update inventory, insert a transaction (new id, amount, balance, timestamp).
4) ALWAYS set:
   - `answer_text` (human sentence, required),
   - `STATUS` (see list above).
   Also print a brief log to stdout, e.g., "LOG: ACTION=read DRY_RUN=True STATUS=no_match".
5) Optional: set `answer_rows` or `answer_json` if useful, but `answer_text` is mandatory.

TONE EXAMPLES (for `answer_text`):
- success: "Yes, we have our Classic sunglasses, a round frame, for $60."
- no_match: "We don’t have round frames under $100 in stock right now, but our Moon round frame is available at $120."
- insufficient_stock: "We only have 1 pair of Classic left; I can reserve that for you."
- invalid_request: "I can help with that—how many pairs would you like to purchase?"
- unsupported_intent: "We can’t refurbish frames, but I can suggest similar new models."

Constraints:
- Use TinyDB Query for filtering. Standard library imports only if needed.
- Keep code clear and commented with numbered steps.

User request:
{question}
"""


### 2.2 From Prompt to Code (Planning in Code)

Let’s generate code that **is the plan**.

Instead of asking the model to output a plan in JSON and running it step-by-step with many tiny tools, let’s have it **write Python that encodes the whole plan** (e.g., “filter this, then compute that, then update this row”). The function `generate_llm_code`:

1. **Builds a live schema** from `inventory_tbl` and `transactions_tbl` so the model sees real fields, types, and examples.
2. **Formats the prompt** with that schema plus the user’s question.
3. **Calls the model** to produce a **plan-with-code** response — typically an `<execute_python>...</execute_python>` block whose body contains the step-by-step logic.
4. **Returns the full response** (including the plan and the code).  
   *We don’t execute anything in this step.*

Why this pattern? Let’s leverage Python/TinyDB as a rich toolbox the model already “knows,” so it can compose multi-step solutions directly in code instead of relying on a growing set of bespoke tools. We’ll extract and run the code in a later step.

In [ ]:
# ---------- 1) Code generation ----------
# 代码生成阶段：
# 作用：
# 调用 LLM，根据用户请求生成 "plan-with-code"（代码形式的执行计划）。
# 注意：
# 这里只负责生成代码，不执行代码。
# 后续 execute_generated_code 函数会负责提取并运行生成的 Python。


def generate_llm_code(
    prompt: str,
    *,
    inventory_tbl,
    transactions_tbl,
    model: str = "gpt-4.1-mini",
    temperature: float = 0.2,
) -> str:
    """
    调用 LLM 生成包含执行计划的 Python 代码。

    输入：
    - prompt:
        用户的自然语言请求，例如：
        "购买两副 Ray-Ban 太阳镜"

    - inventory_tbl:
        TinyDB 库存表

    - transactions_tbl:
        TinyDB 交易记录表

    - model:
        使用的 LLM 模型

    - temperature:
        控制生成随机性，较低值让代码更加稳定。


    流程：
    1. 根据 TinyDB 表结构动态生成 schema 信息。
    2. 将 schema + 用户问题填入 System Prompt。
    3. 调用 LLM。
    4. 返回 LLM 生成的完整文本。


    返回：
    - 完整的 LLM response。
      包括：
      <execute_python>
          Python代码
      </execute_python>

      后续步骤再提取其中的 Python 代码执行。
    """

    # Step 1:
    # 根据当前数据库表动态生成 schema 描述。
    # 让 LLM 知道真实字段、类型和数据示例。
    schema_block = inv_utils.build_schema_block(
        inventory_tbl,
        transactions_tbl
    )


    # Step 2:
    # 将 schema 信息和用户问题填入之前定义好的 Prompt。
    #
    # Prompt 会告诉 LLM：
    # - 如何解析用户意图
    # - 如何使用 TinyDB Query
    # - 什么时候查询
    # - 什么时候修改数据库
    # - 输出格式要求
    prompt = PROMPT.format(
        schema_block=schema_block,
        question=prompt
    )


    # Step 3:
    # 调用 LLM 生成 plan-with-code。
    #
    # system message:
    # 定义模型角色：
    # "生成安全、带注释的 TinyDB 操作代码"
    #
    # user message:
    # 具体任务 Prompt
    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {
                "role": "system",
                "content":
                    "You write safe, well-commented TinyDB code "
                    "to handle data questions and updates."
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
    )


    # Step 4:
    # 获取 LLM 返回文本。
    #
    # 注意：
    # 此时只是生成代码计划，并没有执行 Python。
    content = resp.choices[0].message.content or ""


    # 返回完整 response
    # 后续 execute_generated_code 会：
    # 1. 提取 <execute_python> 内代码
    # 2. 在安全环境执行
    # 3. 操作 TinyDB
    return content

### 2.3 Try a Sample Prompt (Planning-in-Code)

We’ll use the same prompt Andrew used in the lecture:

> **Prompt:** “Do you have any round sunglasses in stock that are under $100?”

Before generating any code, let’s manually inspect the TinyDB tables to see if there are truly *round* frames (word-only match) and what their prices look like. Run the next cell to preview the inventory and highlight items that match the word-only “round” filter.

In [ ]:
Item = Query()                    # Create a Query object to reference fields (e.g., Item.name, Item.description)

# Search the inventory table for documents where either the description OR the name
# contains the word "round" (case-insensitive). The check is done inline:
# - (v or "") ensures we handle None by converting it to an empty string
# - .lower() normalizes case
# - " round " enforces a crude word boundary (won't match "wraparound")
round_sunglasses = inventory_tbl.search(
    (Item.description.test(lambda v: " round " in ((v or "").lower()))) |
    (Item.name.test(        lambda v: " round " in ((v or "").lower())))
)

# Render the results as formatted JSON in the notebook UI
utils.print_html(json.dumps(round_sunglasses, indent=2), title="Inventory Status: Round Sunglasses")

Great — we do have round frames available. From our manual inspection, there are two round styles in stock, but only **one** is **under \$100**. Therefore, the item that satisfies the requirement is:

````python
{
  "item_id": "SG005",
  "name": "Classic",
  "description": "Classic round profile with minimalist metal frames, offering a timeless and versatile style that fits both casual and formal wear.",
  "quantity_in_stock": 10,
  "price": 60
}
````

Now let’s ask the model to **generate a plan in code** that answers Andrew’s prompt (no execution yet).

In [ ]:
# Andrew's prompt from the lecture
prompt_round = "Do you have any round sunglasses in stock that are under $100?"

# Generate the plan-as-code (FULL content; may include <execute_python> tags)
full_content_round = generate_llm_code(
    prompt_round,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="o4-mini",
    temperature=1.0,
)

# Inspect the LLM’s plan + code (no execution here)
utils.print_html(full_content_round, title="Plan with Code (Full Response)")

### 2.4. Define the executor function (run a given plan)

Now we’ll define the function that **takes a plan produced by the model and runs it** safely:

- It **accepts either** the full LLM response (with `<execute_python>…</execute_python>`) **or** raw Python code.
- It **extracts** the executable block when needed.
- It runs the code in a **controlled namespace** (TinyDB tables + safe helpers only).
- It captures **stdout**, **errors**, and the model-set answer variables (`answer_text`, `answer_rows`, `answer_json`).
- It renders **before/after** table snapshots to make side effects explicit.

This is the “executor” that turns a **plan-as-code** into actions and a concise user-facing answer.


(.*?)           # 捕获标签内部内容，非贪婪
re.DOTALL       # 允许 . 匹配换行
re.IGNORECASE   # 忽略标签大小写

re.search(...)

re 是 Python 的正则表达式模块。

re.search(pattern, text, flags)

它的作用是：

在整个 text 中搜索第一个符合 pattern 的部分。

eturn m.group(1).strip() if m else text.strip()

它是 Python 的三元表达式，等价于：

if m:
    return m.group(1).strip()
else:
    return text.strip()

具体看两种情况。

情况 1：m 找到了匹配内容

前面有：

m = re.search(
    r"<execute_python>(.*?)</execute_python>",
    text,
    re.DOTALL | re.IGNORECASE
)

如果 text 是：

hello

<execute_python>
x = 1
print(x)
</execute_python>

那么 m 是一个匹配对象，不是 None。

于是执行：

m.group(1).strip()

其中：

m.group(1)

得到：


x = 1
print(x)


再经过：

.strip()

去掉前后的空格和换行，变成：

x = 1
print(x)

所以最终返回的是标签内部的代码。

情况 2：根本没有 <execute_python>

例如：

text = """
x = 1
print(x)
"""

这时：

m = re.search(...)

找不到，所以：

m is None

那么：

if m

为 False，于是执行：

text.strip()

直接返回原始文本，只是去掉前后空白。

In [ ]:
# --- Helper: extract code between <execute_python>...</execute_python> ---
def _extract_execute_block(text: str) -> str:
    """
    从 LLM 返回的内容中提取 <execute_python>...</execute_python> 标签内的 Python 代码。

    如果没有检测到标签，则默认传入的 text 本身就是纯 Python 代码。
    """
    if not text:
        raise RuntimeError("Empty content passed to code executor.")

    # re.DOTALL: 让 . 可以匹配换行符
    # re.IGNORECASE: 标签大小写不敏感
    m = re.search(
        r"<execute_python>(.*?)</execute_python>",
        text,
        re.DOTALL | re.IGNORECASE
    )

    # 如果有标签，只返回标签内部代码；
    # 否则直接返回原始文本
    return m.group(1).strip() if m else text.strip()

这两步是在给“后面要执行的模型生成代码”准备一个受控的运行环境。 SAFE_GLOBALS │ ├── Query ├── get_current_balance() ├── next_transaction_id() └── user_request

SAFE_LOCALS │ ├── db ├── inventory_tbl └── transactions_tbl

模型代码运行的时候，能访问这些名字。

比如模型生成：

balance = get_current_balance()

if balance > 100: db.execute(...)

这里：

get_current_balance ↑ 来自 SAFE_GLOBALS

db ↑ 来自 SAFE_LOCALS

In [ ]:
# ---------- 2) Code execution ----------

def execute_generated_code(
    code_or_content: str,
    *,
    db,
    inventory_tbl,
    transactions_tbl,
    user_request: Optional[str] = None,
) -> Dict[str, Any]:
    """
    执行 LLM 生成的 Python 代码。

    输入：
    - code_or_content:
        可以是纯 Python 代码，
        也可以是包含 <execute_python> 标签的完整 LLM 输出
    - db:
        TinyDB 数据库对象
    - inventory_tbl:
        库存表
    - transactions_tbl:
        交易记录表
    - user_request:
        用户原始请求，可供生成代码执行时参考

    输出：
    - 实际执行的代码
    - print() 输出
    - 执行错误
    - 代码生成的最终答案
    - 执行后的数据库状态
    """

    # 1. 提取真正需要执行的 Python 代码
    code = _extract_execute_block(code_or_content)

    # 2. 定义允许生成代码访问的全局变量/函数
    #  这两步是在给“后面要执行的模型生成代码”准备一个受控的运行环境。
    SAFE_GLOBALS = {
        "Query": Query,
        "get_current_balance": inv_utils.get_current_balance,
        "next_transaction_id": inv_utils.next_transaction_id,
        "user_request": user_request or "",
    }

    # 3. 提供代码执行时可以访问的数据库对象
    SAFE_LOCALS = {
        "db": db,
        "inventory_tbl": inventory_tbl,
        "transactions_tbl": transactions_tbl,
    }

    # 4. 暂时把 stdout 重定向到 StringIO，
    #    用来捕获生成代码中的 print() 输出
    _stdout_buf = io.StringIO()
    _old_stdout = sys.stdout
    sys.stdout = _stdout_buf

    err_text = None

    try:
        # 5. 执行 LLM 生成的代码，可以说是在“受控的命名空间”里运行，但不能简单说这就是“安全沙箱”。
        # exec() 本身并不是 sandbox。比如如果没有特别限制 __builtins__，生成代码仍可能访问一些 Python 内置能力。
        exec(code, SAFE_GLOBALS, SAFE_LOCALS)

    except Exception:
        # 如果执行出错，保存完整 traceback，而不是让程序直接崩掉
        err_text = traceback.format_exc()

    finally:
        # 无论成功还是失败，都恢复原来的 stdout
        sys.stdout = _old_stdout

    # 6. 获得代码中 print() 出来的文本
    printed = _stdout_buf.getvalue().strip()

    # 7. 尝试从生成代码中读取标准答案变量
    #    LLM 可以选择设置 answer_text / answer_rows / answer_json
    answer = (
        SAFE_LOCALS.get("answer_text")
        or SAFE_LOCALS.get("answer_rows")
        or SAFE_LOCALS.get("answer_json")
    )

    # 8. 返回执行过程和数据库执行后的状态，方便检查/debug
    return {
        "code": code,                            # 实际执行的 Python 代码
        "stdout": printed,                       # print() 输出
        "error": err_text,                       # 异常信息，无异常时为 None
        "answer": answer,                        # LLM 代码生成的最终答案
        "transactions_tbl": transactions_tbl.all(),  # 当前交易记录
        "inventory_tbl": inventory_tbl.all(),         # 当前库存状态
    }

{
    "code": "...",
    "stdout": "查询成功",
    "error": None,
    "answer": "当前库存是 20",
    "transactions_tbl": [...],
    "inventory_tbl": [...]
}

You’ve checked the shelves and confirmed there’s exactly one round style under $100. Now the fun part: let’s hand the model’s plan-as-code to our executor and watch it do the work. The executor will peel out the <code><execute_python>...</execute_python></code> block, run it in a locked-down sandbox, and then show you everything that matters—what changed in the tables (before/after), any logs the plan printed, and the final, customer-friendly answer_text.

这段代码就是在真正调用你前面定义的 execute_generated_code()，执行 LLM 生成的计划，然后只把最终答案显示出来。

full_content_round = """
我会查询库存中价格低于100美元的圆形太阳镜。

<execute_python>
q = Query()
rows = inventory_tbl.search(
    (q.shape == "round") & (q.price < 100)
)

answer_rows = rows
</execute_python>
"""

In [ ]:
# Execute the generated plan for the round-sunglasses question
result = execute_generated_code(
    full_content_round,          # the full LLM response you generated earlier
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    user_request=prompt_round, # e.g., "Do you have any round sunglasses in stock that are under $100?"
)

# Peek at exactly what Python the plan executed
utils.print_html(result["answer"], title="Plan Execution · Extracted Answer")

As you can see, this is the expected result based on our previous manual analysis.

## 2.4 Return Two Aviator Sunglasses

In the previous step we only **queried** the data, so inventory and transactions were unchanged.  
Now let’s handle a **return** scenario using the planning-in-code pattern:
> **Request:** “Return 2 Aviator sunglasses I bought last week.”

Before generating the plan, let’s **inspect the current inventory** for the *Aviator* model.

这次要做一个真正会修改数据库状态的例子——退货。

用户请求是：

“Return 2 Aviator sunglasses I bought last week.”

退掉我上周买的 2 副 Aviator 太阳镜。

这里和前面的查询例子最大的区别是：

前面：

query inventory

只是查数据，所以：

inventory 不变
transactions 不变

这次退货要真的更新数据，所以一般会做类似：

1. 找到 Aviator 这款商品
2. 检查用户上周是否真的买过至少 2 副
3. 库存 +2
4. 新增一条 return/refund transaction
5. 返回处理结果

所以这里说的：

Before generating the plan, let’s inspect the current inventory

意思是：先看看 Aviator 当前库存是多少，再让 LLM 生成执行计划。

比如原来：

Aviator inventory = 5

退货成功后可能变成：

Aviator inventory = 7

同时 transactions_tbl 里增加一条类似：

{
    "type": "return",
    "product": "Aviator",
    "quantity": 2
}

一句话概括：这段代码是在真正执行退货之前，先查询并显示 Aviator 太阳镜当前的库存状态，本身不会修改数据库。

In [ ]:
# 创建 TinyDB 查询对象，用于引用库存表中的字段
Item = Query()

# 在 inventory_tbl 中查询所有名称严格等于 "Aviator" 的库存记录。
# 这里使用的是区分大小写的精确匹配，因此 "aviator" 不会被匹配到；
# 如果需要忽略大小写，可以改用 .test(...) 或 .matches(...) 并结合 re.I。
aviators = inventory_tbl.search(
    Item.name == "Aviator"
)

# 将查询到的 Aviator 库存记录转换成格式化 JSON，
# 再以 HTML 面板的形式显示出来，用于查看退货操作执行前的库存状态。
utils.print_html(
    json.dumps(aviators, indent=2),
    title="Inventory status: Aviator sunglasses before return"
)

Inventory confirms one Aviator SKU in stock — **SG001 (Aviator)**: **23** units at **$80** each. Now let's generate a plan to answer the prompt:

In [ ]:
# 用户请求：退回上周购买的 2 副 Aviator 太阳镜
prompt_aviator = "Return 2 Aviator sunglasses I bought last week."

# 调用 LLM，根据用户请求和当前库存/交易数据生成“计划 + Python代码”。
# full_content_aviator 保存的是 LLM 的完整原始输出，
# 其中可能包含普通文字说明，也可能包含 <execute_python>...</execute_python> 代码块。
# 注意：这里仅生成计划和代码，还没有真正执行退货操作。
full_content_aviator = generate_llm_code(
    prompt_aviator,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="o4-mini",
    temperature=1,
)

# 将 LLM 生成的完整计划和代码显示出来，方便人工检查。
# 这一步只是展示内容，不会执行其中的 Python 代码，也不会修改库存或交易记录。
utils.print_html(
    full_content_aviator,
    title="Plan with Code (Full Response)"
)

以上的的 Plan with Code (Full Response) 不是单独的“计划文本”，而是：

计划说明 + 可执行代码，混合在一起的完整输出。

Before we execute the plan, let’s check the current status of the transactions.

In [ ]:
utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table Before Return")

The transaction log currently shows a single entry — the opening balance (`TXN001`) for `$500.00` recorded at `2025-10-03T09:16:59.628898`.

Ready to go—execute the plan by running the cell below.

In [ ]:
# 执行前面由 LLM 为 Aviator 退货请求生成的完整计划代码。
# execute_generated_code() 会先从 full_content_aviator 中提取
# <execute_python>...</execute_python> 内的 Python 代码，
# 然后在提供的数据库和受控执行环境中真正运行。
# 因此这一步可能会修改 inventory_tbl 和 transactions_tbl。
result = execute_generated_code(
    full_content_aviator,          # LLM 之前生成的完整“计划 + 代码”
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    user_request=prompt_aviator,   # 原始用户请求
)

# 从执行结果 result 中取出最终答案 answer，
# 再以 HTML 形式显示出来。
# 注意：这里的 print_html 只是展示结果，
# 真正的代码执行已经在 execute_generated_code() 中完成。
utils.print_html(
    result["answer"],
    title="Plan Execution · Extracted Answer"
)

You can see below that a new transaction has been inserted for the Aviator sunglasses return.

In [ ]:
utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table After Return")

And by running the cell below, you’ll see the Aviator stock increase to 25 (`quantity_in_stock`).

In [ ]:
Item = Query()

aviators = inventory_tbl.search(
    (Item.name == "Aviator")
)

utils.print_html(json.dumps(aviators, indent=2), title="Inventory status: Aviator sunglasses after return")

## 3. Putting It All Together: Customer Service Agent

You’ve built the pieces—schema, prompt, code generator, and executor. Now let’s wire them up into a single helper that takes a natural-language request, generates a plan-as-code, executes it safely, and shows the result (plus before/after tables).

**What this agent does**
- Optionally reseeds the demo data for a clean run.
- Generates the plan (Python inside `<execute_python>…</execute_python>`).
- Executes the plan in a controlled namespace (TinyDB + helpers).
- Surfaces a concise `answer_text` and renders before/after snapshots.

这段可以整体理解为：把前面“生成代码 → 执行代码 → 查看执行前后数据库 → 返回结果”整个流程封装成一个完整的 customer service agent 函数。
generate_llm_code()
        ↓
full_content
        ↓
execute_generated_code()
        ↓
exec_res
        ↓
查看数据库前后变化

用户问题
  ↓
LLM 生成计划 + Python代码
  ↓
提取 Python代码
  ↓
执行代码，查询/修改数据库
  ↓
返回最终答案

In [ ]:
def customer_service_agent(
    question: str,
    *,
    db,
    inventory_tbl,
    transactions_tbl,
    model: str = "o4-mini",
    temperature: float = 1.0,
    reseed: bool = False,
) -> dict:
    """
    一个端到端的客服 Agent 封装函数：

    1. 可选：重新初始化库存表和交易表
    2. 接收用户问题
    3. 调用 LLM 生成“计划 + Python代码”
    4. 展示执行前的库存和交易记录
    5. 执行 LLM 生成的代码
    6. 展示最终答案以及执行后的数据库状态
    7. 返回完整的 LLM 输出和执行结果，方便后续检查/debug
    """

    # 0. 如果 reseed=True，则重新初始化数据库中的库存和交易数据
    if reseed:
        inv_utils.create_inventory()
        inv_utils.create_transactions()

    # 1. 展示用户输入的问题
    utils.print_html(question, title="User Question")

    # 2. 调用 LLM，根据用户问题和当前数据库状态生成 plan-as-code
    #    full_content 保存 LLM 的完整输出，其中可能包含
    #    <execute_python>...</execute_python> 代码块
    full_content = generate_llm_code(
        question,
        inventory_tbl=inventory_tbl,
        transactions_tbl=transactions_tbl,
        model=model,
        temperature=temperature,
    )

    # 展示 LLM 生成的完整“计划 + 代码”，此时还没有执行
    utils.print_html(
        full_content,
        title="Plan with Code (Full Response)"
    )

    # 3. 在真正执行代码之前，保存并展示当前数据库状态，
    #    方便之后与执行后的状态进行比较
    utils.print_html(
        json.dumps(inventory_tbl.all(), indent=2),
        title="Inventory Table · Before"
    )

    utils.print_html(
        json.dumps(transactions_tbl.all(), indent=2),
        title="Transactions Table · Before"
    )

    # 4. 真正执行 LLM 生成的 Python 代码
    #    内部会：
    #    - 提取 <execute_python> 中的代码
    #    - 在受控 namespace 中 exec()
    #    - 收集 answer / stdout / error 等结果
    #    - 根据生成代码的逻辑，可能修改库存表和交易表
    exec_res = execute_generated_code(
        full_content,
        db=db,
        inventory_tbl=inventory_tbl,
        transactions_tbl=transactions_tbl,
        user_request=question,
    )

    # 5. 展示执行得到的最终答案
    utils.print_html(
        exec_res["answer"],
        title="Plan Execution · Extracted Answer"
    )

    # 展示执行后的库存表和交易表，
    # 用来观察 Agent 是否真正修改了数据库
    utils.print_html(
        json.dumps(inventory_tbl.all(), indent=2),
        title="Inventory Table · After"
    )

    utils.print_html(
        json.dumps(transactions_tbl.all(), indent=2),
        title="Transactions Table · After"
    )

    # 6. 把整个运行过程的重要信息打包返回
    return {
        # LLM 最原始的完整输出
        "full_content": full_content,

        # 代码执行阶段产生的结果
        "exec": {
            "code": exec_res["code"],          # 实际执行的 Python 代码
            "stdout": exec_res["stdout"],      # print/log 输出
            "error": exec_res["error"],        # 报错信息，没有则为 None
            "answer": exec_res["answer"],      # 最终回答

            # 执行完成后的数据库状态
            "inventory_after": inventory_tbl.all(),
            "transactions_after": transactions_tbl.all(),
        },
    }

## 4. Try It Out (with the Customer Service Agent)

Use the `customer_service_agent(...)` helper to go from a natural-language request → plan-as-code → safe execution → before/after snapshots.

**Try these prompts:**
1) **Read-only (Andrew’s example):**  
   “Do you have any round sunglasses in stock that are under $100?”
2) **Mutation — return:**  
   “Return 2 Aviator sunglasses.”
3) **Mutation — purchase:**  
   “Purchase 3 Wayfarer sunglasses for customer Alice.”
4) **Mutation - purchase multiple items:**
   "I want to buy 3 pairs of classic sunglasses and 1 pair of aviator."


<div style="border:1px solid #93c5fd; border-left:6px solid #3b82f6; background:#eff6ff; border-radius:8px; padding:14px 16px; color:#1e3a8a; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">
  🔎 <strong>What does <code>reseed=True</code> do?</strong><br><br>
  When you call <code>customer_service_agent(..., reseed=True)</code>, the agent <em>re-initializes</em> the demo data before running your prompt:
  <ul style="margin:8px 0 0 18px;">
    <li><strong>Resets</strong> the <code>inventory_tbl</code> to the default product set.</li>
    <li><strong>Resets</strong> the <code>transactions_tbl</code> to a single opening-balance entry.</li>
    <li>Ensures a <strong>clean, reproducible</strong> run so results aren’t affected by previous tests.</li>
  </ul>
  Set <code>reseed=False</code> if you want to <strong>preserve</strong> the current state and continue from prior operations.
</div>



In [ ]:
prompt = "I want to buy 3 pairs of classic sunglasses and 1 pair of aviator sunglasses."

out = customer_service_agent(
    prompt,
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="o4-mini",
    temperature=1.0,
    reseed=True,   # set False to keep current state of the inventory and the transactions
)

## 5. Takeaways

- **You let code be the plan.** Following Andrew’s “code-as-action” idea, you had the model write Python that chains the steps (filter → compute → update) and then you just ran it.

- **You skipped the brittle tool soup.** Instead of piling on tiny tools or JSON plans, you used Python/TinyDB—giving the model a big, familiar toolbox that handles many query shapes with one prompt.

- **You kept runs safe and visible.** You executed in a controlled namespace, captured logs/errors, and reviewed before/after tables—so you always know what changed and why.

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 <strong>Congratulations!</strong>

You just finished the lab and built an <em>agentic</em> customer service workflow. You let the model write code as the plan, ran it safely, and used simple validations to keep updates reliable. When things failed, you surfaced clear, human-readable reasons; when things worked, you saw exactly what changed via before/after snapshots.

With this pattern—planning <em>in</em> code, plus transparent execution—you’re ready to design your own workflows that feel automatic, safe, and easy to extend. 🚀

</div>
